In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
from sklearn.preprocessing import LabelEncoder
import warnings, gc
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
import catboost as cb  
from IPython.display import display
pd.set_option('display.max_columns', None)
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder

from tqdm import tqdm

In [10]:
train = pd.read_csv('/kaggle/input/playground-series-s4e12/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s4e12/test.csv')
sub = pd.read_csv('/kaggle/input/playground-series-s4e12/sample_submission.csv')

print("Train Data Details..................\n")
print("#"*130)
print(f"Train Data Shape: {train.shape}")
print("#"*130)
print(f"Train Data Info: {train.info()}")
print("#"*130)
print(f"Check Train Data NUll Values: {train.isnull().sum()}")
print("#"*130)
print(f"Train data description: {train.describe()}")
print("#"*130)
display(train.head())


print("Test Data Details..................\n\n")
print("#"*130)
print(f"Test Data Shape: {test.shape}")
print("#"*130)
print(f"Test Data Info: {test.info()}")
print("#"*130)
print(f"Check Test Data NUll Values: {test.isnull().sum()}")
print("#"*130)
print(f"Test data description: {test.describe()}")
print("#"*130)
display(test.head())

Train Data Details..................

##################################################################################################################################
Train Data Shape: (1200000, 21)
##################################################################################################################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200000 entries, 0 to 1199999
Data columns (total 21 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   id                    1200000 non-null  int64  
 1   Age                   1181295 non-null  float64
 2   Gender                1200000 non-null  object 
 3   Annual Income         1155051 non-null  float64
 4   Marital Status        1181471 non-null  object 
 5   Number of Dependents  1090328 non-null  float64
 6   Education Level       1200000 non-null  object 
 7   Occupation            841925 non-null   object 
 8   Health Score        

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,0,19.0,Female,10049.0,Married,1.0,Bachelor's,Self-Employed,22.598761,Urban,Premium,2.0,17.0,372.0,5.0,2023-12-23 15:21:39.134960,Poor,No,Weekly,House,2869.0
1,1,39.0,Female,31678.0,Divorced,3.0,Master's,NaN,15.569731,Rural,Comprehensive,1.0,12.0,694.0,2.0,2023-06-12 15:21:39.111551,Average,Yes,Monthly,House,1483.0
2,2,23.0,Male,25602.0,Divorced,3.0,High School,Self-Employed,47.177549,Suburban,Premium,1.0,14.0,NaN,3.0,2023-09-30 15:21:39.221386,Good,Yes,Weekly,House,567.0
3,3,21.0,Male,141855.0,Married,2.0,Bachelor's,NaN,10.938144,Rural,Basic,1.0,0.0,367.0,1.0,2024-06-12 15:21:39.226954,Poor,Yes,Daily,Apartment,765.0
4,4,21.0,Male,39651.0,Single,1.0,Bachelor's,Self-Employed,20.376094,Rural,Premium,0.0,8.0,598.0,4.0,2021-12-01 15:21:39.252145,Poor,Yes,Weekly,House,2022.0


Test Data Details..................


##################################################################################################################################
Test Data Shape: (800000, 20)
##################################################################################################################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    800000 non-null  int64  
 1   Age                   787511 non-null  float64
 2   Gender                800000 non-null  object 
 3   Annual Income         770140 non-null  float64
 4   Marital Status        787664 non-null  object 
 5   Number of Dependents  726870 non-null  float64
 6   Education Level       800000 non-null  object 
 7   Occupation            560875 non-null  object 
 8   Health Score          750551 non-n

,id,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Policy Start Date,Customer Feedback,Smoking Status,Exercise Frequency,Property Type
0,1200000,28.0,Female,2310.0,NaN,4.0,Bachelor's,Self-Employed,7.657981,Rural,Basic,NaN,19.0,NaN,1.0,2023-06-04 15:21:39.245086,Poor,Yes,Weekly,House
1,1200001,31.0,Female,126031.0,Married,2.0,Master's,Self-Employed,13.381379,Suburban,Premium,NaN,14.0,372.0,8.0,2024-04-22 15:21:39.224915,Good,Yes,Rarely,Apartment
2,1200002,47.0,Female,17092.0,Divorced,0.0,PhD,Unemployed,24.354527,Urban,Comprehensive,NaN,16.0,819.0,9.0,2023-04-05 15:21:39.134960,Average,Yes,Monthly,Condo
3,1200003,28.0,Female,30424.0,Divorced,3.0,PhD,Self-Employed,5.136225,Suburban,Comprehensive,1.0,3.0,770.0,5.0,2023-10-25 15:21:39.134960,Poor,Yes,Daily,House
4,1200004,24.0,Male,10863.0,Divorced,2.0,High School,Unemployed,11.844155,Suburban,Premium,NaN,14.0,755.0,7.0,2021-11-26 15:21:39.259788,Average,No,Weekly,House


# PreProcess & Clean Data

In [11]:
train.drop(columns=["id"],axis=1,inplace=True)

def clean_data(df):
    df=df.copy()
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Annual Income'].fillna(df['Annual Income'].mean(), inplace=True)
    df['Marital Status'].fillna(df['Marital Status'].mode()[0], inplace=True)
    df['Number of Dependents'].fillna(df['Number of Dependents'].median(), inplace=True)
    df['Occupation'].fillna(df['Occupation'].mode()[0], inplace=True)
    df['Health Score'].fillna(df['Health Score'].mean(), inplace=True)
    df['Previous Claims'].fillna(df['Previous Claims'].mean(), inplace=True)
    df['Vehicle Age'].fillna(df['Vehicle Age'].mode()[0], inplace=True)
    df['Credit Score'].fillna(df['Credit Score'].mean(), inplace=True)
    df['Insurance Duration'].fillna(df['Insurance Duration'].median(), inplace=True)
    df['Customer Feedback'].fillna(df['Customer Feedback'].mode()[0], inplace=True)
    return df

train=clean_data(train)
test=clean_data(test)

In [12]:
def ultimate_feature_engineering(df, is_train=True):
    df = df.copy()
    
    df['Policy Start Date'] = pd.to_datetime(df['Policy Start Date'], errors='coerce')
    df['year']       = df['Policy Start Date'].dt.year
    df['month']      = df['Policy Start Date'].dt.month
    df['day']        = df['Policy Start Date'].dt.day
    df['dow']        = df['Policy Start Date'].dt.dayofweek
    df['is_weekend'] = (df['dow'] >= 5).astype(int)
    
    df['log_income']            = np.log1p(df['Annual Income'])
    df['income_per_age']        = df['Annual Income'] / (df['Age'] + 1)
    df['income_per_dependent'] = df['Annual Income'] / (df['Number of Dependents'] + 1)
    df['high_income']           = (df['Annual Income'] > 100000).astype(int)
    
    df['age_group']      = pd.cut(df['Age'], bins=[0, 25, 35, 50, 100], labels=[0,1,2,3]).astype(int)
    df['age_x_income']   = df['Age'] * df['log_income']
    df['age_x_dependents'] = df['Age'] * df['Number of Dependents']
    
    df['smoker']        = (df['Smoking Status'] == 'Yes').astype(int)
    df['no_exercise']   = (df['Exercise Frequency'] == 'Never').astype(int)
    df['risk_score']    = df['smoker']*3 + df['no_exercise']*2 + (df['Health Score'] < 20).astype(int)*2
    
    df['credit_group']   = pd.qcut(df['Credit Score'], q=10, labels=False, duplicates='drop')
    df['new_car']        = (df['Vehicle Age'] <= 3).astype(int)
    df['long_duration']  = (df['Insurance Duration'] >= 5).astype(int)
    df['many_claims']    = (df['Previous Claims'] >= 3).astype(int)
    df['premium_policy'] = (df['Policy Type'] == 'Premium').astype(int)
    
    df['income_x_risk']     = df['log_income'] * df['risk_score']
    df['age_x_risk']        = df['Age'] * df['risk_score']
    df['credit_x_income']   = df['Credit Score'] * df['log_income']
    df['health_x_income']   = df['Health Score'] * df['log_income']
    
    freq_cols = ['Gender','Marital Status','Education Level','Occupation','Location',
                 'Policy Type','Property Type','Smoking Status','Exercise Frequency']
    for col in freq_cols:
        df[f'{col}_freq'] = df[col].map(df[col].value_counts(normalize=True))
    
    df = df.drop(columns=['Policy Start Date','Customer Feedback'], errors='ignore')
    
    return df


train=ultimate_feature_engineering(train,is_train=True)
test=ultimate_feature_engineering(test,is_train=False)

In [13]:
def fit_label_encoders(train_df):
    le_dict = {}
    for col in tqdm(train_df.columns, desc="Encoding train columns"):
        if train_df[col].dtype == 'object' or train_df[col].dtype.name == 'category':
            le = LabelEncoder()
            train_df[col] = train_df[col].astype(str)
            le.fit(train_df[col])
            le_dict[col] = le
            train_df[col] = le.transform(train_df[col])
    return train_df, le_dict

def transform_with_encoders(df, le_dict):
    df_encoded = df.copy()
    for col in tqdm(le_dict.keys(), desc="Encoding test columns"):
        if col in df_encoded.columns:
            df_encoded[col] = df_encoded[col].astype(str)
            df_encoded[col] = df_encoded[col].map(lambda s: le_dict[col].transform([s])[0] if s in le_dict[col].classes_ else -1)
    return df_encoded

train, encoders = fit_label_encoders(train)
test = transform_with_encoders(test, encoders)


Encoding test columns: 100%|██████████| 9/9 [04:55<00:00, 32.79s/it]


In [14]:
target = train['Premium Amount']
train = train.drop(['Premium Amount'], axis=1)
test = test.drop(columns=['id'], axis=1)

In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

pred_xgb = np.zeros(len(test))
pred_lgb = np.zeros(len(test))
pred_cat = np.zeros(len(test))


for fold, (ti, vi) in enumerate(kf.split(train)):
    X_tr, X_va = train.iloc[ti], train.iloc[vi]
    y_tr, y_va = np.log1p(target.iloc[ti]), np.log1p(target.iloc[vi])

    mx = xgb.XGBRegressor(n_estimators=10000, learning_rate=0.015, max_depth=11,
                          subsample=0.85, colsample_bytree=0.75,
                          tree_method='gpu_hist', gpu_id=0, predictor='gpu_predictor',
                          random_state=42, verbosity=0)
    
    mx.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=200, verbose=False)

    ml = lgb.LGBMRegressor(n_estimators=10000, learning_rate=0.015, max_depth=12, num_leaves=1024,
                           subsample=0.8, colsample_bytree=0.7, device='gpu',
                           random_state=42, verbose=-1)

    
    ml.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(200)])

    mc = cb.CatBoostRegressor(iterations=10000, learning_rate=0.02, depth=10,
                           l2_leaf_reg=5, task_type='GPU', devices='0',
                           random_state=42, verbose=False)
    mc.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=300, verbose=False)

    oof_xgb[vi] = np.expm1(mx.predict(X_va))
    oof_lgb[vi] = np.expm1(ml.predict(X_va))
    oof_cat[vi] = np.expm1(mc.predict(X_va))

    pred_xgb += np.expm1(mx.predict(test)) / 10
    pred_lgb += np.expm1(ml.predict(test)) / 10
    pred_cat += np.expm1(mc.predict(test)) / 10
    print(f"=== Fold {fold+1}/5 DONE ===\n")

def score(w):
    p = w[0]*oof_xgb + w[1]*oof_lgb + w[2]*oof_cat
    return mean_squared_log_error(target, np.clip(p, 0, None)) ** 0.5

opt = minimize(score, [0.33, 0.33, 0.34], method='Nelder-Mead', bounds=[(0,1)]*3)
w = opt.x / opt.x.sum()

print(f"XGB {w[0]:.4f} | LGB {w[1]:.4f} | Cat {w[2]:.4f} | CV {opt.fun:.6f}")

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[263]	valid_0's l2: 1.11033
=== Fold 1/5 DONE ===

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[238]	valid_0's l2: 1.10827
=== Fold 2/5 DONE ===

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[265]	valid_0's l2: 1.10994
=== Fold 3/5 DONE ===

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[237]	valid_0's l2: 1.10562
=== Fold 4/5 DONE ===

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[355]	valid_0's l2: 1.10916
=== Fold 5/5 DONE ===

XGB 0.2376 | LGB 0.4899 | Cat 0.2725 | CV 1.052531


In [17]:
final=w[0]*pred_xgb + w[1]*pred_lgb + w[2]*pred_cat
sub = pd.read_csv('/kaggle/input/playground-series-s4e12/sample_submission.csv')
sub['Premium Amount'] = final
sub.to_csv('submission.csv', index=False)
sub.head()

,id,Premium Amount
0,1200000,340.590123
1,1200001,406.751977
2,1200002,395.526323
3,1200003,400.258066
4,1200004,379.367193
